# Sprint 3 Campus — Stream and observe FieldCare

A technician wants to read an answer as it arrives. Support wants to find the corresponding request afterward. We will connect those two views without putting private request content in logs.

Use **one saved copy of this notebook throughout the sprint**. Work one demonstration at a time: predict, run, read the result, then explain the boundary it reveals. Each section names the matching Campus lesson. The optional project cells stay off until you are ready to test your own work; **Run all runs the supplied demos without requiring a finished assessment**.

| Campus lesson | Notebook destination | What to notice |
|---|---|---|
| When streaming helps | §2 · Add the streaming operation and safe observation | Choose delivery by the caller’s needs. |
| Enabling streaming on the scaffold | §2–4 · Stream and buffered comparison | Partial text needs a terminal outcome. |
| What to log and why | §5 · Inspect the safe operational record | Match exact request IDs to terminal records. |
| What NOT to log | §5 · Check the allowlist against synthetic sensitive fields | Keep operational metadata and exclude content. |
| Closing the loop with existing evaluation | §6 · Reuse the original evaluation criteria | Choose approved cases from an observed concern. |
| Observable streaming service checkpoint | §7 · Checkpoint and carry forward | Apply the patterns to your own service and defend the evidence. |

**Two folders, two purposes.** `DEMO` contains supplied, runnable instructor examples. `PROJECT` contains your editable service and any imported checkpoint. Demonstrations never edit `PROJECT`; a successful demo is not assessment completion. Source modules and the evaluation design are already in the repository. Cells make calls and inspect results; they do not manufacture Python files.

**Before running:** add `OPENROUTER_API_KEY` to Colab Secrets and enable notebook access, or use the hidden prompt. Initial Chroma indexing and supported model requests use provider quota. Generated wording and timings vary. Keep credentials out of cells and output.

Open Colab’s table of contents for section navigation. Save your personal copy in Drive for notes and outputs; use the final checkpoint cell to save your own service files.

## Campus lessons

- [When streaming helps](https://app.notion.com/p/3ea5a55972ad81fd9cdee92887860939?pvs=204)
- [Enabling streaming on the scaffold](https://app.notion.com/p/3ea5a55972ad8118a84ef55a5d48442e?pvs=204)
- [What to log and why](https://app.notion.com/p/3ea5a55972ad817f865ec69c602dd137?pvs=204)
- [What NOT to log](https://app.notion.com/p/3ea5a55972ad8168bb14fe42cd0c288f?pvs=204)
- [Closing the loop with existing evaluation](https://app.notion.com/p/3ea5a55972ad816f89b7f0bd61f6d601?pvs=204)
- [Observable streaming service checkpoint](https://app.notion.com/p/3ea5a55972ad81d9b6dafb3e76ff75fd?pvs=204)

## 1. Set up the same service

The supplied demo has streaming, authentication, and observation already registered in repository files. To continue your own service, set `CHECKPOINT` to your uploaded Sprint 2 ZIP and choose a new `PROJECT` directory. Setup keeps it separate from the demo and makes no source edits.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys
import json

REPOSITORY = "https://github.com/richhiey/ai-app-dev_Mod-B.git"
BASE = Path("/content") if Path("/content").exists() else Path.cwd() / "campus-runtime"
REPO = BASE / "ai-app-dev_Mod-B"
DEMO = BASE / "fieldcare-campus-demo-3"
PROJECT = BASE / "fieldcare-campus-project-3"
CHECKPOINT = ""  # Optional earlier Campus source ZIP; restore to a new PROJECT path.

if not REPO.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", "main", REPOSITORY, str(REPO)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO), "-c", str(REPO / "requirements.lock")], check=True)
sys.path.insert(0, str(REPO / "src"))

from module_b.openrouter import DEFAULT_CHAT_MODEL, require_openrouter_key
from module_b.workspace import prepare_example
from module_b.campus import prepare_project, save_checkpoint
require_openrouter_key(prompt=True)
os.environ.setdefault("OPENROUTER_MODEL", DEFAULT_CHAT_MODEL)
os.environ["CAMPUS_SPRINT"] = "3"
DEMO = prepare_example(destination=DEMO, repo_root=REPO)
PROJECT = prepare_project(REPO, PROJECT, CHECKPOINT)
os.chdir(DEMO)
sys.path.insert(0, str(DEMO))

print("Instructor source:", DEMO)
print("Your project:", PROJECT)
print("Project origin:", "imported checkpoint" if CHECKPOINT else "supplied starter (not a completed assessment)")

## 2. Add the streaming operation and safe observation

### Decide what the caller needs

A technician screen can render partial text. A ticket integration may need one complete object before it proceeds. Add your two delivery decisions to the text cell below; this is a design decision, not a speed contest.

### Read the supplied implementation

Open `DEMO/app/stream_routes.py` and `DEMO/app/campus_demo.py`. Trace the request schema → clarification → LangGraph retrieval → `provider_events` → `yield encode_event(event)`. The supplied router is registered before the caller guard inventories the protected paths. The observation middleware is outermost so it also sees authentication and limiter rejections.

```python
app.include_router(stream_router)
# After all routers: attach SecurityMiddleware with protected_post_paths(app).
# Add ObservationMiddleware last so it observes both rejected and admitted requests.
```

This demo uses a four-request allowance per caller to leave room for the boundary checks. It does not alter the two-request policy in your own project. `ServiceProcess` runs an owned loopback Uvicorn process; there is no public tunnel or foreground server cell.

### Your delivery decisions

**Technician screen:** choose buffered or streamed delivery and state what the user can do before completion.

**Ticket integration:** choose buffered or streamed delivery and state what evidence allows the caller to treat the result as complete.

In [ ]:
import secrets
import time
import httpx
from module_b.runtime import ServiceProcess
from module_b.observability import FIELDS, safe_record
from module_b.campus import read_request_records
from app.campus_demo import CALLER_ENV
caller_keys = {label: secrets.token_urlsafe(32) for label in CALLER_ENV}
caller_env = {variable: caller_keys[label] for label, variable in CALLER_ENV.items()}
LOG_PATH = DEMO / "logs" / "requests.jsonl"
if globals().get("service") is not None:
    service.stop()
service = ServiceProcess(
    "app.campus_demo:create_app", factory=True, project_dir=DEMO,
    env={**caller_env, "CAMPUS_SPRINT": "3", "CAMPUS_ALLOWANCE": "4", "FIELDCARE_LOG_PATH": str(LOG_PATH)},
    startup_timeout=120,
).start()
payload = {"question": "Which filter and airflow checks are documented?", "equipment_id": "EQ-FC-1002"}
observed_ids = []

### Check the boundary before asking for a stream

**Predict:** a missing key returns `401`; a recognized caller with an empty body returns `422`; a valid question without equipment context returns deterministic JSON clarification. None should produce provider deltas. We keep their request IDs so §5 can match outcomes to logs.

In [ ]:
with httpx.Client(base_url=service.base_url, timeout=30) as client:
    for label, headers, body in [
        ("missing key", {}, payload),
        ("invalid body", {"X-API-Key": caller_keys["dispatch"]}, {}),
        ("missing equipment context", {"X-API-Key": caller_keys["partner"]}, {"question": "Which filter checks are documented?"}),
    ]:
        result = client.post("/v1/diagnose-stream", json=body, headers=headers)
        observed_ids.append(result.headers["x-request-id"])
        print(label, result.status_code, result.headers["content-type"], result.json())
        if label == "missing equipment context":
            clarification_id = result.headers["x-request-id"]

## 3. Read actual incremental provider events

**Predict the client contract:** `metadata` identifies the response; `delta` carries new text; `usage` may report tokens; `complete` confirms success. An `error` means partial text is not a finished answer. HTTP `200` alone does not prove completion after streaming has begun.

Run the request and watch text arrive. Events are forwarded from the real OpenRouter stream as newline-delimited JSON (NDJSON). We record client arrival times; these include retrieval, provider, network, and local overhead. We do not split a prewritten answer into fake chunks.

In [ ]:
events, arrival_seconds = [], []
started = time.perf_counter()
with httpx.Client(base_url=service.base_url, timeout=120) as client:
    with client.stream("POST", "/v1/diagnose-stream", json=payload,
                       headers={"X-API-Key": caller_keys["dispatch"]}) as response:
        request_id = response.headers.get("x-request-id")
        if request_id:
            observed_ids.append(request_id)
        print("HTTP:", response.status_code, "Request ID:", request_id)
        if "application/x-ndjson" not in response.headers.get("content-type", ""):
            response.read()
            print("JSON response (not a provider stream):", response.json())
        else:
            for line in response.iter_lines():
                if line:
                    event = json.loads(line)
                    events.append(event)
                    arrival_seconds.append(time.perf_counter() - started)
                    if event.get("type") == "delta":
                        print(event["text"], end="", flush=True)
print()
terminal = next((e for e in reversed(events) if e.get("type") in {"complete", "error"}), None)
usage = next((e for e in events if e.get("type") == "usage"), None)
first_delta_seconds = next((t for e, t in zip(events, arrival_seconds) if e.get("type") == "delta"), None)
stream_elapsed = time.perf_counter() - started
print("Event types:", list(dict.fromkeys(e.get("type") for e in events)))
print("First text seconds:", first_delta_seconds, "Total client seconds:", round(stream_elapsed, 2))
print("Terminal:", terminal)
print("Provider tokens:", usage.get("tokens") if usage else None, "(None = unknown)")

## 4. Compare with the buffered response

Send the same question with Partner’s independent allowance. The buffered client receives one complete JSON object; generated wording may differ because this is a second real call.

**Compare:** what can each client display before completion, and what signals a finished result? One pair of timings cannot establish a general speed advantage or model quality.

In [ ]:
started = time.perf_counter()
with httpx.Client(base_url=service.base_url, timeout=120) as client:
    response = client.post("/v1/diagnose", json=payload,
                           headers={"X-API-Key": caller_keys["partner"]})
buffered_elapsed = time.perf_counter() - started
buffered_id = response.headers["x-request-id"]
observed_ids.append(buffered_id)
print("Buffered HTTP:", response.status_code, "Request ID:", buffered_id)
print(json.dumps(response.json(), indent=2))
print("Buffered complete response seconds:", round(buffered_elapsed, 2))
print("Stream first text seconds:", first_delta_seconds)
print("Stream ended successfully:", terminal is not None and terminal.get("type") == "complete")

## 5. Inspect the safe operational record

The service wrote these records while handling our requests. This file write has a teaching purpose: it is the operational logging we are studying. It contains allowlisted metadata, not a Python implementation assembled by a notebook.

**Predict:** which records should show `unauthorized` and `invalid_request`? Locate the stream and buffered response using their exact runtime request IDs. Missing model/token values mean unknown, not zero. Only records belonging to this run are selected, so older log entries cannot be mistaken for new evidence.

In [ ]:
records = read_request_records(LOG_PATH, observed_ids)
for record in records:
    print({field: record.get(field) for field in FIELDS})
print("Stream record:", next((r for r in records if r["request_id"] == request_id), None))
print("Buffered record:", next(r for r in records if r["request_id"] == buffered_id))
print("Prompt retained:", any("prompt" in row for row in records))
print("Headers retained:", any("headers" in row for row in records))

### Check the allowlist against synthetic sensitive fields

Start with the stream attempt’s actual safe record, then add a synthetic prompt and header to an in-memory copy. **Predict:** which fields survive? This marker is not a credential and is never sent to OpenRouter or written to a log.

In [ ]:
marker = "SYNTHETIC-NOT-A-CREDENTIAL"
observed = next(record for record in records if record["request_id"] == request_id)
candidate = {**observed, "_approved_models": (os.environ["OPENROUTER_MODEL"],),
             "prompt": "Synthetic private marker " + marker,
             "headers": {"X-API-Key": marker}}
minimized = safe_record(candidate, routes=("/v1/diagnose", "/v1/diagnose-stream"))
print("Based on actual request:", minimized["request_id"])
print("Retained fields:", minimized)
print("Marker in retained record:", marker in repr(minimized))
print("Original candidate still holds marker:", marker in repr(candidate))

**Interpret:** minimization preserves the correlation ID and validated operational fields. It does not erase the original object, inspect another logging system, or certify all possible strings. Keep the original candidate out of any output or file that might later be shared.

## 6. Reuse the original evaluation criteria

### From an observation to a case worth rerunning

Our boundary check returned clarification for missing equipment context. Find its terminal log by `clarification_id`. The log tells us which attempt completed without provider work; the paired HTTP response tells us *why*. Metadata alone cannot reconstruct its private question.

We select approved synthetic Module A case `EVAL-FC-005` for missing-context behavior, with `EVAL-FC-003` as a companion for HX/VX evidence selection. Read the original expectations before evaluating. This uses the preserved **Module A deterministic reference pipeline** and the supplied `data/pipeline_design.json`; it does not score generated prose or prove that the live OpenRouter path uses every reference-design setting.

In [ ]:
from module_b._module_a import project as original
from module_b.evaluation import load_pipeline, evaluate_selected
print("Observed concern:", next(r for r in records if r["request_id"] == clarification_id))
design_path = DEMO / "data" / "pipeline_design.json"
pipeline = load_pipeline(design_path)
SELECTED_CASE = "EVAL-FC-005"
COMPANION_CASE = "EVAL-FC-003"
case_ids = [SELECTED_CASE, COMPANION_CASE]
if SELECTED_CASE == COMPANION_CASE:
    raise ValueError("Choose two different approved cases.")
for case_id in case_ids:
    case = original.eval_case_by_id(pipeline.env, case_id)
    print("Original criteria:", case["eval_id"], case["expected_behavior"])
report = evaluate_selected(design_path, case_ids)
for row in report["rows"]:
    print({name: row[name] for name in ["eval_id", "pipeline_pass", "missing_required_docs", "missing_required_tools", "missing_expected_flags"]})
print("Reference design revision:", report["design_revision"])

**Explain:** a log directs investigation; an approved case makes the relevant behavior repeatable. A passed case checks its stated criteria only. It does not grade stream delivery, privacy, provider uptime, or the truth of generated text. For your project, justify both the chosen case and the limits of the comparison.

## 7. Checkpoint and carry forward

### Your project: enable the patterns you observed

Continue in `PROJECT` with the secured Sprint 2 source. The streaming source is provided at `DEMO/app/stream_routes.py` (also in the repository under `examples/patterns/sprint_3/stream_routes.py`). Copy that supplied file with the Files panel into your app if it is absent. Register its router **before** the guard computes protected paths. Add `ObservationMiddleware` last, using a project log path and the approved model list:

```python
import os
from app.config import openrouter_model
from module_b.observability import ObservationMiddleware
from module_b.security import protected_post_paths

app.add_middleware(
    ObservationMiddleware,
    path=os.environ["FIELDCARE_LOG_PATH"],
    routes=protected_post_paths(app),
    approved_models=(openrouter_model(),),
)
```

The project runner supplies `FIELDCARE_LOG_PATH` in the service environment. Read `DEMO/app/campus_demo.py` as the registration reference; preserve your own routers and policy.

Use the Campus assessment brief: choose a supported question different from the demo, predict the event sequence, correlate a real terminal record, explain excluded fields, and choose an approved evaluation case plus a companion. If your imported project lacks `data/pipeline_design.json`, copy the supplied reference file from the repository and identify it as the Module A reference design; do not claim it automatically configures your live model.

The optional cell below tests **your** `app.main`. Put predictions and your interpretation in text cells, then retain its displayed safe record as evidence. It does not install or repair your implementation.

In [ ]:
RUN_MY_PROJECT = False
MY_QUESTION = ""  # Choose a supported question different from the guided example.
if RUN_MY_PROJECT:
    if not MY_QUESTION.strip():
        raise ValueError("Choose your assessment question first.")
    project_log = PROJECT / "logs" / "requests.jsonl"
    with ServiceProcess(project_dir=PROJECT, env={**caller_env, "FIELDCARE_LOG_PATH": str(project_log)},
                        startup_timeout=120) as project_service:
        with httpx.Client(base_url=project_service.base_url, timeout=120) as project_client:
            with project_client.stream("POST", "/v1/diagnose-stream",
                    json={"question": MY_QUESTION, "equipment_id": "EQ-FC-1002"},
                    headers={"X-API-Key": caller_keys["dispatch"]}) as result:
                project_request_id = result.headers.get("x-request-id")
                print("Project HTTP:", result.status_code, "Request ID:", project_request_id)
                for line in result.iter_lines():
                    if line:
                        print(json.loads(line))
        if not project_request_id:
            raise ValueError("No request ID: inspect your observation middleware before claiming completion.")
        print("Project terminal record:", read_request_records(project_log, [project_request_id]))
else:
    print("Assessment has not run. Apply the patterns to your project before enabling this cell.")

### Your evaluation handoff

Use an operational concern from your run to justify two approved cases. The IDs below are intentionally blank: select them yourself. The evaluator remains a reference-pipeline check, separate from your live-service and generated-answer evidence.

In [ ]:
MY_CASE_IDS = []  # One selected case and one companion, with your reasons in a text cell.
if RUN_MY_PROJECT:
    if len(MY_CASE_IDS) != 2 or len(set(MY_CASE_IDS)) != 2:
        raise ValueError("Choose two distinct approved evaluation cases.")
    project_report = evaluate_selected(PROJECT / "data" / "pipeline_design.json", MY_CASE_IDS)
    print(json.dumps(project_report, indent=2))
else:
    print("Project evaluation awaits your case selection.")

## Save your Campus checkpoint

Save the notebook in Drive to retain predictions, outputs, and notes. The export below packages **your `PROJECT` source and data**, not the instructor demo. If you only ran the demos, it contains an unchanged starter. Review your editable files before sharing; an export allowlist cannot detect a credential pasted into Python code.

Download the printed ZIP using Colab’s Files panel. In the next sprint, set `CHECKPOINT` to its uploaded path and use a new `PROJECT` folder. For a same-runtime rerun after restoring, clear `CHECKPOINT` to reuse the existing project. Never overwrite an existing workspace to recover from an error.

In [ ]:
if globals().get("service") is not None:
    service.stop()
CHECKPOINT_OUT = save_checkpoint(PROJECT, BASE / "exports", sprint=3)
print("Your project checkpoint:", CHECKPOINT_OUT)
print("Save the notebook separately for observations and any safe log records you displayed.")

## Watch out for

- **Missing package after setup:** run setup in a fresh runtime. If this runtime holds an old source checkout, preserve your project ZIP and use Runtime → Disconnect and delete runtime, then reopen this same notebook link.
- **Provider error or quota limit:** keep the actual error, check your provider configuration, and rerun only the affected call. Do not substitute a sample answer.
- **Changing Python files has no effect:** stop the owned process and restart it. The supplied demo and your `app.main` are different applications; check which one the request cell starts.
- **Checkpoint destination already exists:** clear `CHECKPOINT` to reuse it, or choose a new `PROJECT` path to restore. Your existing files are preserved.

## Summary

Use the observations above to explain this sprint’s service boundaries. Complete your own project tasks using the lesson brief, then keep its source and run evidence together for the next sprint.